In [1]:
import lightgbm as lgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import top_k_accuracy_score as top_k

In [11]:
y_true = np.array([0, 1, 2, 1])

y_score = np.array([[0.5, 0.2, 0.2],  # 0 is in top 2
                    [0.3, 0.4, 0.2],  # 1 is in top 2
                    [0.2, 0.4, 0.3],  # 2 is in top 2
                    [0.7, 0.2, 0.1]]) # 2 isn't in top 2

top_k(y_true, y_score, k=1)

0.5

In [10]:
a = [3,7,9,2, 1, 5, 8, 6, 9, 15]
np.histogram(a, bins=[0,10,20], density=False)

(array([9, 1], dtype=int64), array([ 0, 10, 20]))

In [12]:
def load_embeddings(datapath, filename):
    f = open(datapath+filename, 'r', encoding='utf8')
    emb_map = {}
    for line in f:
        ls = line.split()
        vals = [float(val) for val in ls[1:]]
        if len(vals) == 1:
            continue
        emb_map[ls[0]] = vals
    f.close()
    return emb_map

def load_labels(datapath, fname):
    f = open(datapath+fname, 'r', encoding='utf8')
    pairs = []
    for line in f:
        ls = line.split()
        pairs.append((ls[0], ls[1]))
    return pairs

def read_entities_map(datapath, filename):
    uri_map = {}
    f = open(datapath+filename, 'r', encoding="utf8")
    for line in f:
        nr_id, uri = line.split()[0], line.split()[1]
        uri_map[nr_id] = uri
    f.close()
    return uri_map

In [13]:
datapath_embeddings = 'data/embeddings/'
entity_embs1 = load_embeddings(datapath_embeddings, 'final_embs1.txt')
entity_embs2 = load_embeddings(datapath_embeddings, 'final_embs2.txt')

In [14]:
datapath = 'data/fr_en/'
ref_pairs = load_labels(datapath, 'ref_pairs')
sup_pairs = load_labels(datapath, 'sup_pairs')

In [15]:
ent_map1 = read_entities_map(datapath, 'ent_ids_1')
ent_map2 = read_entities_map(datapath, 'ent_ids_2')

In [16]:
list(entity_embs1.keys())[:10]

['Rodrigo_Rato',
 'Mariano_Rajoy',
 'Robert_Schuman',
 'Georges_Bidault',
 'Ed_Miliband',
 'David_Cameron',
 'Call_of_Duty_(série)',
 'Wii_U',
 'Nothing_but_the_Beat',
 'I_Can_Only_Imagine']

In [ ]:
def pairs_to_X_y(pairs, ent_map1, ent_map2, entity_embs1, entity_embs2, nr_neg=5):
    feats, y = [], []
    for pair in pairs:
        p1, p2 = ent_map1[pair[0]].split('/')[-1], ent_map2[pair[1]].split('/')[-1]
        feats.append(list(entity_embs1[p1]) + list(entity_embs2[p2]))
        y.append(1)

    if nr_neg > 0:
        vals1, vals2 = list(entity_embs1.values()), list(entity_embs2.values())
        rs1, rs2 = np.random.choice(list(range(len(vals1))), size=nr_neg*len(y)), np.random.choice(list(range(len(vals2))), size=nr_neg*len(y))

        for idx1, idx2 in zip(rs1, rs2):
            feats.append(list(vals1[idx1]) + list(vals2[idx2]))
            y.append(0)

    assert len(feats) == len(y)
    X = pd.DataFrame(np.array(feats))

    return X, y

In [158]:
def entity_pairs(pairs, ent_map2, entity_embs2):
    ents2, embs2 = [], []
    for pair in pairs:
        p2 = ent_map2[pair[1]].split('/')[-1]
        ents2.append(p2)
        embs2.append(entity_embs2[p2])
    return ents2, embs2

def get_feature_matrix(pair, ent_map1, ent_map2, entity_embs1, ents2, embs2):
    feats = []
    print(pair)
    p1, p2 = ent_map1[pair[0]].split('/')[-1], ent_map2[pair[1]].split('/')[-1]
    emb2 = entity_embs1[p1]
    index = ents2.index(p2)
    for emb2 in embs2:
        feats.append(list(entity_embs1[p1]) + list(emb2))
    return pd.DataFrame(feats), index
        

query = 0
ents2, embs2 = entity_pairs(ref_pairs[:1000], ent_map2, entity_embs2)
all_feats, index = get_feature_matrix(ref_pairs[query], ent_map1, ent_map2, entity_embs1, ents2, embs2)


('0', '10500')


In [159]:
preds = gbm.predict(all_feats)
len(preds)

1000

In [160]:
print(ent_map1[ref_pairs[query][0]], ent_map2[ref_pairs[query][1]])
for idx in preds.argsort()[-10:][::-1]:
    print(idx, ents2[idx])

http://fr.dbpedia.org/resource/Saint-Joseph-de-Coleraine http://dbpedia.org/resource/Saint-Joseph-de-Coleraine,_Quebec
290 Sainte-Anne-de-la-Rochelle,_Quebec
420 Notre-Dame-de-la-Salette,_Quebec
36 Lac-Ministuk,_Quebec
316 Lemieux,_Quebec
720 Kazabazua,_Quebec
901 Lac-Nilgaut,_Quebec
836 L'Épiphanie,_Quebec_(parish)
539 Bowman,_Quebec
95 Saint-Michel,_Quebec
500 Stornoway,_Quebec


In [119]:
sorted(preds, reverse=True)[:20]

[0.9015677777637239,
 0.8220218782341923,
 0.8006091727670446,
 0.7891112572341675,
 0.7828030058723846,
 0.7777901802827958,
 0.7774880657548554,
 0.774829231527021,
 0.7721667640929738,
 0.7689763896297573,
 0.7639642046675894,
 0.7625857685803248,
 0.7624209374847385,
 0.7598202453418675,
 0.7586210586703543,
 0.7530348690808991,
 0.7529287306252007,
 0.7528486051932796,
 0.7526255193880053,
 0.7503653777057107]

In [25]:
# feats, y = [], []
# for pair in sup_pairs:
#     p1, p2 = ent_map1[pair[0]].split('/')[-1], ent_map2[pair[1]].split('/')[-1]
#     feats.append(list(entity_embs1[p1]) + list(entity_embs2[p2]))
#     y.append(1)

# vals1, vals2 = list(entity_embs1.values()), list(entity_embs2.values())
# rs1, rs2 = np.random.choice(list(range(len(vals1))), size=5*len(y)), np.random.choice(list(range(len(vals2))), size=5*len(y))

# for idx1, idx2 in zip(rs1, rs2):
#     feats.append(list(vals1[idx1]) + list(vals2[idx2]))
#     y.append(0)




In [31]:
# X = np.array(feats)
X, y = pairs_to_X_y(sup_pairs, ent_map1, ent_map2,  entity_embs1, entity_embs2)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=73)

print('train/validation', X_train.shape,X_val.shape, np.mean(y_train), np.mean(y_val), len(y_train), len(y_val))

# X_test, y_test = pairs_to_X_y(ref_pairs, ent_map1, ent_map2, nr_neg=0)

# print('test', X_test.shape, np.mean(y_test))

train/validation (18900, 3600) (8100, 3600) 0.16687830687830688 0.16617283950617284 18900 8100


In [ ]:
lgb_train = lgb.Dataset(pd.DataFrame(X_train), y_train)
lgb_eval = lgb.Dataset(pd.DataFrame(X_val), y_val, reference=lgb_train)

In [21]:
params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": {"auc", "average_precision"},
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": 1,
}

gbm = lgb.train(
    params, lgb_train, num_boost_round=1000, valid_sets=lgb_eval, callbacks=[lgb.early_stopping(stopping_rounds=25), \
                                                                             lgb.log_evaluation(period=20, show_stdv=True)]
)

[LightGBM] [Info] Number of positive: 3154, number of negative: 15746
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.395114 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 788037
[LightGBM] [Info] Number of data points in the train set: 18900, number of used features: 3600
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.166878 -> initscore=-1.607915
[LightGBM] [Info] Start training from score -1.607915
Training until validation scores don't improve for 25 rounds
[20]	valid_0's auc: 0.893293	valid_0's average_precision: 0.671852
[40]	valid_0's auc: 0.921665	valid_0's average_precision: 0.744088
[60]	valid_0's auc: 0.937085	valid_0's average_precision: 0.782654
[80]	valid_0's auc: 0.950628	valid_0's average_precision: 0.814093
[100]	valid_0's auc: 0.958038	valid_0's average_precision: 0.834769
[120]	valid_0's auc: 0.962492	valid_0's average_precision: 0.84467
[140]	valid_0's auc: 0.966145	valid_0's

In [28]:
preds = gbm.predict(X_test)

print(len(preds))

10500


In [134]:
bst = lgb.Booster(model_file="models/model.txt")

In [142]:
new = bst.predict(all_feats)

In [141]:
preds

array([0.00917936, 0.71244531, 0.00606375, ..., 0.04294131, 0.03192974,
       0.03668412])

In [146]:
from scipy.stats import pearsonr
pearsonr(new, preds)

PearsonRResult(statistic=0.8847634044562214, pvalue=0.0)